# Collect results (Machine Translation)

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from utils import detrend_ue
from pathlib import Path
import pathlib


methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
    'eurollm': 'EuroLLM 9B',
}

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen', 'wmt14_deen', 'wmt14_ruen', 'wmt14_fren',
        'wmt19_deen', 'wmt19_fien', 'wmt19_lten', 'wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen', 'wmt14_deen', 'wmt14_ruen', 'wmt14_fren',
        'wmt19_deen', 'wmt19_fien', 'wmt19_lten', 'wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen', 'wmt14_deen', 'wmt14_ruen', 'wmt14_fren',
        'wmt19_deen', 'wmt19_fien', 'wmt19_lten', 'wmt19_ruen',
    ]
}

METRICS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': 'MetricX XXL',
    'XComet-XCOMET-XXL': 'XComet XXL',
    'Comet-wmt22-comet-da': 'Comet WMT22',
}


def main():
    rows = []  # will accumulate dicts with columns: model, dataset, metric, method, prr_score

    for metric_key, metric_name in METRICS.items():
        datasets = DATASETS[metric_key]
        ue_methods = list(methods_dict.values())

        for model_key, model_name in MODELS.items():
            # compute UE scores (raw + detrended), same as before
            ue_scores, _, _ = detrend_ue(
                datasets,
                model_key,
                [metric_key],
                ue_methods,
                methods_dict
            )

            # flatten into rows
            for method_full, method_short in methods_dict.items():
                raw_scores = ue_scores[f"{method_short}_raw"]
                detr_scores = ue_scores[f"{method_short}_detr"]

                for dataset_name, raw, detr in zip(datasets, raw_scores, detr_scores):
                    # raw row (method name unchanged)
                    rows.append({
                        "model": model_name,
                        "dataset": dataset_name,
                        "metric": metric_name,
                        "method": method_short,
                        "prr_score": float(raw),
                    })
                    # detrended row (method-LINE)
                    rows.append({
                        "model": model_name,
                        "dataset": dataset_name,
                        "metric": metric_name,
                        "method": f"{method_short}-LINE",
                        "prr_score": float(detr),
                    })

    # write single CSV with all metrics/models/datasets/methods
    df = pd.DataFrame(rows, columns=["model", "dataset", "metric", "method", "prr_score"])
    out_path = Path("results") / "mt_results.csv"
    df.to_csv(out_path, index=False)
    print(f"Wrote {out_path.resolve()} with {len(df)} rows.")

if __name__ == "__main__":
    main()


In [4]:
import pandas as pd


df = pd.read_csv("results/mt_results.csv")


import numpy as np
from IPython.display import display, Markdown


methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}
method_order = [v for v in methods_dict.values()]
method_display_order = [m for pair in zip(method_order, [m + "-LINE" for m in method_order]) for m in pair]

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
    'eurollm': 'EuroLLM 9B',
}
model_display_order = list(MODELS.values())

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen','wmt14_deen','wmt14_ruen','wmt14_fren',
        'wmt19_deen','wmt19_fien','wmt19_lten','wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen','wmt14_deen','wmt14_ruen','wmt14_fren',
        'wmt19_deen','wmt19_fien','wmt19_lten','wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen','wmt14_deen','wmt14_ruen','wmt14_fren',
        'wmt19_deen','wmt19_fien','wmt19_lten','wmt19_ruen',
    ]
}

# ---- highlighter: best = bold, second = underline (ties handled) ----
def highlight_best_second(col: pd.Series):
    if col.dtype.kind not in "fi":
        return [''] * len(col)
    vals = col.astype(float)
    uniq_sorted = np.unique(vals[~vals.isna()])[::-1]  # desc unique
    best = uniq_sorted[0] if len(uniq_sorted) > 0 else np.nan
    second = uniq_sorted[1] if len(uniq_sorted) > 1 else np.nan

    styles = []
    for v in vals:
        if pd.isna(v):
            styles.append('')
        elif np.isclose(v, best, rtol=1e-9, atol=1e-12):
            styles.append('font-weight: bold;')
        elif not np.isnan(second) and np.isclose(v, second, rtol=1e-9, atol=1e-12):
            styles.append('text-decoration: underline;')
        else:
            styles.append('')
    return styles

# ---- render one "page" of tables per metric ----
def show_metric_tables(df_in: pd.DataFrame, metric_name: str):
    display(Markdown(f"## {metric_name}"))
    metric_df = df_in[df_in["metric"] == metric_name].copy()

    # choose a pleasant dataset column order
    candidate_order = None
    for ds_list in DATASETS.values():
        if set(ds_list).issubset(set(metric_df["dataset"].unique())):
            candidate_order = ds_list
            break
    if candidate_order is None:
        candidate_order = list(metric_df["dataset"].unique())

    for model_name in model_display_order:
        block = metric_df[metric_df["model"] == model_name]
        if block.empty:
            continue

        piv = block.pivot_table(
            index="method", columns="dataset", values="prr_score", aggfunc="mean"
        )
        # reindex to desired display ordering
        piv = piv.reindex(index=method_display_order)
        piv = piv.reindex(columns=[c for c in candidate_order if c in piv.columns])

        styled = (
            piv.round(3)
               .style
               .apply(highlight_best_second, axis=0)
               .set_caption(model_name)
        )
        display(styled)

# ---- run for each metric ----
for metric_name in df["metric"].unique():
    show_metric_tables(df, metric_name)


## MetricX XXL

dataset,wmt14_csen,wmt14_deen,wmt14_ruen,wmt14_fren,wmt19_deen,wmt19_fien,wmt19_lten,wmt19_ruen
method,,,,,,,,
MSP,0.215000,0.225000,0.313000,0.194000,0.225000,0.084000,0.119000,0.264000
MSP-LINE,0.396000,0.422000,0.436000,0.310000,0.403000,0.412000,0.382000,0.355000
PPL,0.429000,0.450000,0.414000,0.319000,0.390000,0.482000,0.434000,0.311000
PPL-LINE,0.484000,0.473000,0.485000,0.357000,0.404000,0.482000,0.436000,0.396000
MTE,0.475000,0.477000,0.459000,0.393000,0.427000,0.520000,0.487000,0.356000
MTE-LINE,0.537000,0.514000,0.538000,0.435000,0.471000,0.509000,0.489000,0.454000
MCSE,0.157000,0.140000,0.202000,0.163000,0.143000,-0.002000,0.065000,0.213000
MCSE-LINE,0.292000,0.291000,0.304000,0.246000,0.318000,0.295000,0.288000,0.275000
MCNSE,0.423000,0.378000,0.404000,0.324000,0.383000,0.393000,0.438000,0.324000


dataset,wmt14_csen,wmt14_deen,wmt14_ruen,wmt14_fren,wmt19_deen,wmt19_fien,wmt19_lten,wmt19_ruen
method,,,,,,,,
MSP,0.189000,0.219000,0.288000,0.133000,0.279000,0.058000,0.241000,0.265000
MSP-LINE,0.392000,0.446000,0.412000,0.290000,0.446000,0.352000,0.349000,0.386000
PPL,0.425000,0.471000,0.405000,0.326000,0.419000,0.408000,0.331000,0.340000
PPL-LINE,0.429000,0.479000,0.427000,0.330000,0.427000,0.407000,0.342000,0.371000
MTE,0.450000,0.475000,0.419000,0.359000,0.440000,0.454000,0.335000,0.350000
MTE-LINE,0.463000,0.495000,0.460000,0.371000,0.467000,0.446000,0.373000,0.414000
MCSE,0.110000,0.155000,0.213000,0.120000,0.201000,-0.027000,0.172000,0.229000
MCSE-LINE,0.308000,0.359000,0.328000,0.276000,0.401000,0.266000,0.245000,0.352000
MCNSE,0.383000,0.431000,0.402000,0.330000,0.429000,0.363000,0.319000,0.381000


dataset,wmt14_csen,wmt14_deen,wmt14_ruen,wmt14_fren,wmt19_deen,wmt19_fien,wmt19_lten,wmt19_ruen
method,,,,,,,,
MSP,0.127000,0.226000,0.297000,0.111000,0.230000,0.041000,0.155000,0.284000
MSP-LINE,0.324000,0.444000,0.445000,0.265000,0.428000,0.339000,0.312000,0.379000
PPL,0.512000,0.521000,0.454000,0.415000,0.481000,0.445000,0.403000,0.334000
PPL-LINE,0.515000,0.529000,0.458000,0.419000,0.483000,0.436000,0.423000,0.371000
MTE,0.541000,0.537000,0.477000,0.456000,0.498000,0.486000,0.424000,0.361000
MTE-LINE,0.548000,0.547000,0.470000,0.465000,0.512000,0.470000,0.467000,0.420000
MCSE,0.200000,0.237000,0.283000,0.164000,0.247000,0.098000,0.254000,0.273000
MCSE-LINE,0.417000,0.421000,0.389000,0.360000,0.402000,0.357000,0.356000,0.343000
MCNSE,0.230000,0.330000,0.295000,0.223000,0.276000,0.324000,0.264000,0.214000


## XComet XXL

...

dataset,wmt14_csen,wmt14_deen,wmt14_ruen,wmt14_fren,wmt19_deen,wmt19_fien,wmt19_lten,wmt19_ruen
method,,,,,,,,
MSP,0.133000,0.308000,0.379000,0.205000,0.268000,-0.027000,0.058000,0.355000
MSP-LINE,0.237000,0.384000,0.425000,0.248000,0.383000,0.261000,0.189000,0.365000
PPL,0.394000,0.405000,0.375000,0.328000,0.425000,0.397000,0.331000,0.314000
PPL-LINE,0.412000,0.435000,0.499000,0.373000,0.437000,0.384000,0.336000,0.390000
MTE,0.434000,0.417000,0.389000,0.345000,0.430000,0.448000,0.386000,0.322000
MTE-LINE,0.462000,0.462000,0.511000,0.417000,0.473000,0.421000,0.381000,0.440000
MCSE,0.185000,0.305000,0.348000,0.263000,0.303000,-0.009000,0.080000,0.337000
MCSE-LINE,0.307000,0.365000,0.383000,0.331000,0.384000,0.279000,0.200000,0.322000
MCNSE,0.191000,0.277000,0.229000,0.171000,0.259000,0.300000,0.225000,0.212000


## Comet WMT22

dataset,wmt14_csen,wmt14_deen,wmt14_ruen,wmt14_fren,wmt19_deen,wmt19_fien,wmt19_lten,wmt19_ruen
method,,,,,,,,
MSP,0.424000,0.394000,0.453000,0.349000,0.458000,0.186000,0.288000,0.431000
MSP-LINE,0.469000,0.487000,0.481000,0.398000,0.511000,0.467000,0.473000,0.414000
PPL,0.417000,0.455000,0.371000,0.314000,0.407000,0.516000,0.475000,0.315000
PPL-LINE,0.521000,0.515000,0.526000,0.412000,0.459000,0.521000,0.495000,0.464000
MTE,0.445000,0.479000,0.406000,0.374000,0.418000,0.540000,0.518000,0.331000
MTE-LINE,0.576000,0.563000,0.589000,0.484000,0.546000,0.561000,0.555000,0.526000
MCSE,0.365000,0.322000,0.347000,0.296000,0.361000,0.085000,0.201000,0.359000
MCSE-LINE,0.376000,0.356000,0.328000,0.280000,0.385000,0.318000,0.355000,0.320000
MCNSE,0.481000,0.435000,0.401000,0.358000,0.432000,0.455000,0.483000,0.353000


dataset,wmt14_csen,wmt14_deen,wmt14_ruen,wmt14_fren,wmt19_deen,wmt19_fien,wmt19_lten,wmt19_ruen
method,,,,,,,,
MSP,0.398000,0.373000,0.425000,0.286000,0.490000,0.185000,0.350000,0.401000
MSP-LINE,0.482000,0.501000,0.473000,0.383000,0.533000,0.422000,0.360000,0.412000
PPL,0.436000,0.484000,0.382000,0.361000,0.445000,0.456000,0.304000,0.305000
PPL-LINE,0.457000,0.506000,0.496000,0.400000,0.466000,0.456000,0.321000,0.355000
MTE,0.443000,0.489000,0.384000,0.371000,0.436000,0.487000,0.302000,0.304000
MTE-LINE,0.492000,0.537000,0.529000,0.436000,0.506000,0.493000,0.360000,0.401000
MCSE,0.319000,0.308000,0.353000,0.283000,0.406000,0.092000,0.287000,0.361000
MCSE-LINE,0.394000,0.431000,0.375000,0.352000,0.472000,0.307000,0.294000,0.387000
MCNSE,0.442000,0.502000,0.417000,0.370000,0.473000,0.413000,0.352000,0.373000


dataset,wmt14_csen,wmt14_deen,wmt14_ruen,wmt14_fren,wmt19_deen,wmt19_fien,wmt19_lten,wmt19_ruen
method,,,,,,,,
MSP,0.293000,0.332000,0.423000,0.238000,0.401000,0.162000,0.277000,0.430000
MSP-LINE,0.374000,0.461000,0.500000,0.342000,0.507000,0.394000,0.341000,0.436000
PPL,0.505000,0.505000,0.434000,0.440000,0.516000,0.484000,0.357000,0.325000
PPL-LINE,0.531000,0.531000,0.537000,0.473000,0.536000,0.486000,0.403000,0.394000
MTE,0.525000,0.517000,0.456000,0.466000,0.511000,0.512000,0.372000,0.341000
MTE-LINE,0.572000,0.553000,0.564000,0.520000,0.575000,0.521000,0.450000,0.453000
MCSE,0.351000,0.359000,0.424000,0.283000,0.414000,0.211000,0.343000,0.422000
MCSE-LINE,0.462000,0.466000,0.456000,0.396000,0.487000,0.400000,0.375000,0.389000
MCNSE,0.225000,0.357000,0.282000,0.221000,0.338000,0.355000,0.277000,0.245000


# Collect results (Summarization and Math Reasoning)

In [ ]:
from utils import detrend_ue_w_quality


MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
}

DATASETS = {
    'Accuracy': [
        'gsm8k'
    ],
    'AlignScoreInputOutput': [
        'xsum'
    ]
}

METRICS = {
    'Accuracy': 'Accuracy',
    'AlignScoreInputOutput': 'Align Score',
}


def main():
    rows = []  # will accumulate dicts with columns: model, dataset, metric, method, prr_score

    for metric_key, metric_name in METRICS.items():
        datasets = DATASETS[metric_key]
        ue_methods = list(methods_dict.values())

        for model_key, model_name in MODELS.items():
            # compute UE scores (raw + detrended), same as before
            ue_scores, _, _ = detrend_ue_w_quality(
                datasets,
                model_key,
                [metric_key],
                ue_methods,
                methods_dict
            )

            # flatten into rows
            for method_full, method_short in methods_dict.items():
                raw_scores = ue_scores[f"{method_short}_raw"]
                detr_scores = ue_scores[f"{method_short}_detr"]

                for dataset_name, raw, detr in zip(datasets, raw_scores, detr_scores):
                    # raw row (method name unchanged)
                    rows.append({
                        "model": model_name,
                        "dataset": dataset_name,
                        "metric": metric_name,
                        "method": method_short,
                        "prr_score": float(raw),
                    })
                    # detrended row (method-LINE)
                    rows.append({
                        "model": model_name,
                        "dataset": dataset_name,
                        "metric": metric_name,
                        "method": f"{method_short}-LINE",
                        "prr_score": float(detr),
                    })

    # write single CSV with all metrics/models/datasets/methods
    df = pd.DataFrame(rows, columns=["model", "dataset", "metric", "method", "prr_score"])
    out_path = Path("results") / "sum_mr_results.csv"
    df.to_csv(out_path, index=False)
    print(f"Wrote {out_path.resolve()} with {len(df)} rows.")

if __name__ == "__main__":
    main()


In [3]:


df = pd.read_csv("results/sum_mr_results.csv")


import numpy as np
from IPython.display import display, Markdown


methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}
method_order = [v for v in methods_dict.values()]
method_display_order = [m for pair in zip(method_order, [m + "-LINE" for m in method_order]) for m in pair]

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
}
model_display_order = list(MODELS.values())


DATASETS = {
    'Accuracy': [
        'gsm8k'
    ],
    'AlignScoreInputOutput': [
        'xsum'
    ]
}

METRICS = {
    'Accuracy': 'Accuracy',
    'AlignScoreInputOutput': 'Align Score',
}

# ---- highlighter: best = bold, second = underline (ties handled) ----
def highlight_best_second(col: pd.Series):
    if col.dtype.kind not in "fi":
        return [''] * len(col)
    vals = col.astype(float)
    uniq_sorted = np.unique(vals[~vals.isna()])[::-1]  # desc unique
    best = uniq_sorted[0] if len(uniq_sorted) > 0 else np.nan
    second = uniq_sorted[1] if len(uniq_sorted) > 1 else np.nan

    styles = []
    for v in vals:
        if pd.isna(v):
            styles.append('')
        elif np.isclose(v, best, rtol=1e-9, atol=1e-12):
            styles.append('font-weight: bold;')
        elif not np.isnan(second) and np.isclose(v, second, rtol=1e-9, atol=1e-12):
            styles.append('text-decoration: underline;')
        else:
            styles.append('')
    return styles

# ---- render one "page" of tables per metric ----
def show_metric_tables(df_in: pd.DataFrame, metric_name: str):
    display(Markdown(f"## {metric_name}"))
    metric_df = df_in[df_in["metric"] == metric_name].copy()

    # choose a pleasant dataset column order
    candidate_order = None
    for ds_list in DATASETS.values():
        if set(ds_list).issubset(set(metric_df["dataset"].unique())):
            candidate_order = ds_list
            break
    if candidate_order is None:
        candidate_order = list(metric_df["dataset"].unique())

    for model_name in model_display_order:
        block = metric_df[metric_df["model"] == model_name]
        if block.empty:
            continue

        piv = block.pivot_table(
            index="method", columns="dataset", values="prr_score", aggfunc="mean"
        )
        # reindex to desired display ordering
        piv = piv.reindex(index=method_display_order)
        piv = piv.reindex(columns=[c for c in candidate_order if c in piv.columns])

        styled = (
            piv.round(3)
               .style
               .apply(highlight_best_second, axis=0)
               .set_caption(model_name)
        )
        display(styled)

# ---- run for each metric ----
for metric_name in df["metric"].unique():
    show_metric_tables(df, metric_name)


## Accuracy

dataset,gsm8k
method,
MSP,0.324000
MSP-LINE,0.329000
PPL,0.303000
PPL-LINE,0.377000
MTE,0.339000
MTE-LINE,0.400000
MCSE,0.351000
MCSE-LINE,0.352000
MCNSE,0.343000


dataset,gsm8k
method,
MSP,0.303000
MSP-LINE,0.296000
PPL,0.248000
PPL-LINE,0.358000
MTE,0.292000
MTE-LINE,0.399000
MCSE,0.393000
MCSE-LINE,0.399000
MCNSE,0.356000


## Align Score

dataset,xsum
method,
MSP,0.328000
MSP-LINE,0.357000
PPL,0.369000
PPL-LINE,0.366000
MTE,0.357000
MTE-LINE,0.350000
MCSE,0.033000
MCSE-LINE,0.043000
MCNSE,0.024000


dataset,xsum
method,
MSP,0.351000
MSP-LINE,0.378000
PPL,0.354000
PPL-LINE,0.373000
MTE,0.333000
MTE-LINE,0.356000
MCSE,0.003000
MCSE-LINE,0.032000
MCNSE,0.016000


# Results with smaller sample size

In [1]:
from utils import detrend_ue_w_quality

import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path


methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
}

DATASETS = {
    'Accuracy': [
        'gsm8k'
    ],
    'AlignScoreInputOutput': [
        'xsum'
    ]
}

METRICS = {
    'Accuracy': 'Accuracy',
    'AlignScoreInputOutput': 'Align Score',
}


def main():
    rows = []  # will accumulate dicts with columns: model, dataset, metric, method, prr_score

    for metric_key, metric_name in METRICS.items():
        datasets = DATASETS[metric_key]
        ue_methods = list(methods_dict.values())

        for model_key, model_name in MODELS.items():
            # compute UE scores (raw + detrended), same as before
            ue_scores, _, _ = detrend_ue_w_quality(
                datasets,
                model_key,
                [metric_key],
                ue_methods,
                methods_dict,
                quality_fit_sample_size=500
            )

            # flatten into rows
            for method_full, method_short in methods_dict.items():
                raw_scores = ue_scores[f"{method_short}_raw"]
                detr_scores = ue_scores[f"{method_short}_detr"]

                for dataset_name, raw, detr in zip(datasets, raw_scores, detr_scores):
                    # raw row (method name unchanged)
                    rows.append({
                        "model": model_name,
                        "dataset": dataset_name,
                        "metric": metric_name,
                        "method": method_short,
                        "prr_score": float(raw),
                    })
                    # detrended row (method-LINE)
                    rows.append({
                        "model": model_name,
                        "dataset": dataset_name,
                        "metric": metric_name,
                        "method": f"{method_short}-LINE",
                        "prr_score": float(detr),
                    })

    # write single CSV with all metrics/models/datasets/methods
    df = pd.DataFrame(rows, columns=["model", "dataset", "metric", "method", "prr_score"])
    out_path = Path("results") / "sum_mr_results_500.csv"
    df.to_csv(out_path, index=False)
    print(f"Wrote {out_path.resolve()} with {len(df)} rows.")

if __name__ == "__main__":
    main()


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a Bert

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f1d2a2c3670>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f1d2a32d4e0>]
llama gsm8k Below q ids: 1796


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


...

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f1d2a2c2e00>]
llama xsum Below q ids: 1620


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f1d2a2dc4c0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f1d20324f10>]
gemma xsum Below q ids: 1781
Wrote /home/maiya.goloburda/detrend/detrending_ue/results/sum_mr_results_500.csv with 48 rows.


In [2]:
import pandas as pd


sample_500 = pd.read_csv("results/sum_mr_results_500.csv")

sample_full = pd.read_csv("results/sum_mr_results.csv")



import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# expects these to already exist:
# sample_full, sample_500  with columns: model, dataset, metric, method, prr_score

# ---- split out the three variants ----
cols = ["model","dataset","metric","method","prr_score"]
full_raw  = sample_full.loc[~sample_full["method"].str.endswith("-LINE"), cols].copy()
full_detr = sample_full.loc[ sample_full["method"].str.endswith("-LINE"), cols].copy()
s500_detr = sample_500.loc[ sample_500["method"].str.endswith("-LINE"), cols].copy()

# normalize to base method name (no suffix)
for df in (full_detr, s500_detr):
    df["method_base"] = df["method"].str.replace("-LINE","", regex=False)
full_raw["method_base"] = full_raw["method"]

# label/ordering inside each base method block: raw -> 500 -> full
full_raw["variant"]  = "raw"
s500_detr["variant"] = "500"
full_detr["variant"] = "full"

full_raw["method_display"]  = full_raw["method_base"]
s500_detr["method_display"] = s500_detr["method_base"] + "-LINE (500 sample)"
full_detr["method_display"] = full_detr["method_base"] + "-LINE (Full sample)"

full_raw["variant_order"]  = 0
s500_detr["variant_order"] = 1
full_detr["variant_order"] = 2

combined = pd.concat([full_raw, s500_detr, full_detr], ignore_index=True)

# if you want a specific method order, set it here (else inferred from data)
method_order = ["MSP","PPL","MTE","MCSE","MCNSE","LSRL"]
present_methods = [m for m in method_order if m in combined["method_base"].unique()]
# fall back to whatever is present if list is empty
if not present_methods:
    present_methods = list(combined["method_base"].unique())

# ---- highlighter: best bold, second underline (per column) ----
def highlight_best_second(col: pd.Series):
    if col.dtype.kind not in "fi":
        return [''] * len(col)
    vals = col.astype(float)
    uniq = np.unique(vals[~vals.isna()])[::-1]
    best = uniq[0] if len(uniq) else np.nan
    second = uniq[1] if len(uniq) > 1 else np.nan
    out = []
    for v in vals:
        if pd.isna(v): out.append('')
        elif np.isclose(v, best): out.append('font-weight: bold;')
        elif not np.isnan(second) and np.isclose(v, second): out.append('text-decoration: underline;')
        else: out.append('')
    return out

# ---- render per-model blocks ----
for model in combined["model"].unique():
    block = combined[combined["model"] == model].copy()

    # choose dataset column order (feel free to hardcode if you like)
    dataset_order = sorted(block["dataset"].unique())

    # pivot to rows=method_display (ordered as raw→500→full within each base method)
    block["method_base"] = pd.Categorical(block["method_base"], categories=present_methods, ordered=True)
    block = block.sort_values(["method_base","variant_order"])

    piv = (block
           .pivot_table(index=["method_base","variant_order","method_display"],
                        columns="dataset", values="prr_score", aggfunc="mean")
           .reindex(columns=dataset_order)
           .sort_index(level=[0,1])
          )

    # collapse to a single index showing just the display name we want
    piv = piv.reset_index().set_index("method_display").drop(columns=["method_base","variant_order"])
    styled = (piv.round(4)
                 .style
                 .apply(highlight_best_second, axis=0)
                 .set_caption(model))

    display(Markdown(f"### {model}"))
    display(styled)



/tmp/ipykernel_4024775/702947106.py:77: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  piv = (block


### Llama 3.1 8B

dataset,gsm8k,xsum
method_display,,
MSP,0.324400,0.327600
MSP-LINE (500 sample),0.331800,0.357400
MSP-LINE (Full sample),0.328900,0.356900
PPL,0.302600,0.369100
PPL-LINE (500 sample),0.377200,0.366100
PPL-LINE (Full sample),0.376500,0.365600
MTE,0.339300,0.356900
MTE-LINE (500 sample),0.391400,0.353700
MTE-LINE (Full sample),0.399800,0.350400


/tmp/ipykernel_4024775/702947106.py:77: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  piv = (block


### Gemma 2 9B

dataset,gsm8k,xsum
method_display,,
MSP,0.302800,0.350700
MSP-LINE (500 sample),0.294700,0.378200
MSP-LINE (Full sample),0.296400,0.377700
PPL,0.248300,0.354100
PPL-LINE (500 sample),0.357000,0.372000
PPL-LINE (Full sample),0.357600,0.372600
MTE,0.291500,0.333200
MTE-LINE (500 sample),0.399200,0.355600
MTE-LINE (Full sample),0.398600,0.356000
